# Feature Engineering

Notebook này sử dụng các hàm đã được viết trong `src/features/build_features.py` để trích xuất các đặc trưng mới từ dữ liệu thô và mã hóa chúng thành dạng số (phù hợp cho mô hình học máy).

In [1]:
import pandas as pd
import sys
import os

# Thiết lập đường dẫn tương thích mọi môi trường (Jupyter Root hoặc Notebooks Folder)
current_dir = os.getcwd()
if os.path.basename(current_dir) == 'notebooks':
    project_root = os.path.abspath(os.path.join(current_dir, '..'))
else:
    project_root = current_dir

src_path = os.path.join(project_root, 'src')
sys.path.insert(0, src_path)

from features.build_features import extract_date_features, encode_categorical

print("Modules loaded successfully.")

ModuleNotFoundError: No module named 'features'

In [ ]:
# Tải dữ liệu ban đầu
data_path = os.path.join(project_root, 'data', 'raw', 'ecommerce_dataset_updated.csv')
df = pd.read_csv(data_path)
print(f"Kích thước dữ liệu gốc: {df.shape}")
df.head()

Kích thước dữ liệu gốc: (3660, 8)


,User_ID,Product_ID,Category,Price (Rs.),Discount (%),Final_Price(Rs.),Payment_Method,Purchase_Date
0,337c166f,f414122f-e,Sports,36.53,15,31.05,Net Banking,12-11-2024
1,d38a19bf,fde50f9c-5,Clothing,232.79,20,186.23,Net Banking,09-02-2024
2,d7f5f0b0,0d96fc90-3,Sports,317.02,25,237.76,Credit Card,01-09-2024
3,395d4994,964fc44b-d,Toys,173.19,25,129.89,UPI,01-04-2024
4,a83c145c,d70e2fc6-e,Beauty,244.80,20,195.84,Net Banking,27-09-2024


### 1. Trích xuất đặc trưng Thời gian (Date Features)
Cột `Purchase_Date` sẽ được phân tích thành Năm, Tháng, Ngày và Thứ trong tuần.

In [ ]:
# Định dạng ngày tháng là DD-MM-YYYY theo như dữ liệu gốc
df_time_features = extract_date_features(df, date_column='Purchase_Date', date_format="%d-%m-%Y")

# Kiểm tra các cột mới được tạo
df_time_features[['Purchase_Date', 'Purchase_Year', 'Purchase_Month', 'Purchase_Day', 'Purchase_DayOfWeek', 'Purchase_IsWeekend']].head()

,Purchase_Date,Purchase_Year,Purchase_Month,Purchase_Day,Purchase_DayOfWeek,Purchase_IsWeekend
0,2024-11-12,2024,11,12,1,0
1,2024-02-09,2024,2,9,4,0
2,2024-09-01,2024,9,1,6,1
3,2024-04-01,2024,4,1,0,0
4,2024-09-27,2024,9,27,4,0


### 2. Mã hóa các biến Phân loại (Categorical Encoding)
Mã hóa các cột như `Category` và `Payment_Method` thành các cột số (0 hoặc 1) bằng One-Hot Encoding.

In [ ]:
categorical_cols = ['Category', 'Payment_Method']

# Tiến hành mã hóa
df_encoded = encode_categorical(df_time_features, categorical_cols)

print(f"Kích thước dữ liệu sau khi mã hóa: {df_encoded.shape}")
df_encoded.head()

Kích thước dữ liệu sau khi mã hóa: (3660, 21)


,User_ID,Product_ID,Price (Rs.),Discount (%),Final_Price(Rs.),Purchase_Date,Purchase_Year,Purchase_Month,Purchase_Day,Purchase_DayOfWeek,...,Category_Books,Category_Clothing,Category_Electronics,Category_Home & Kitchen,Category_Sports,Category_Toys,Payment_Method_Credit Card,Payment_Method_Debit Card,Payment_Method_Net Banking,Payment_Method_UPI
0,337c166f,f414122f-e,36.53,15,31.05,2024-11-12,2024,11,12,1,...,0,0,0,0,1,0,0,0,1,0
1,d38a19bf,fde50f9c-5,232.79,20,186.23,2024-02-09,2024,2,9,4,...,0,1,0,0,0,0,0,0,1,0
2,d7f5f0b0,0d96fc90-3,317.02,25,237.76,2024-09-01,2024,9,1,6,...,0,0,0,0,1,0,1,0,0,0
3,395d4994,964fc44b-d,173.19,25,129.89,2024-04-01,2024,4,1,0,...,0,0,0,0,0,1,0,0,0,1
4,a83c145c,d70e2fc6-e,244.80,20,195.84,2024-09-27,2024,9,27,4,...,0,0,0,0,0,0,0,0,1,0


### 3. Loại bỏ các cột không cần thiết và Lưu kết quả
Cột `User_ID` và `Product_ID` là định danh, không có tác dụng dự đoán doanh số nên có thể loại bỏ. Cột `Purchase_Date` dạng thời gian cũng có thể loại bỏ vì đã trích xuất xong đặc trưng.

In [ ]:
columns_to_drop = ['User_ID', 'Product_ID', 'Purchase_Date']
df_final = df_encoded.drop(columns=columns_to_drop)

# Xem cấu trúc cuối cùng
df_final.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3660 entries, 0 to 3659
Data columns (total 18 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   Price (Rs.)                 3660 non-null   float64
 1   Discount (%)                3660 non-null   int64  
 2   Final_Price(Rs.)            3660 non-null   float64
 3   Purchase_Year               3660 non-null   int32  
 4   Purchase_Month              3660 non-null   int32  
 5   Purchase_Day                3660 non-null   int32  
 6   Purchase_DayOfWeek          3660 non-null   int32  
 7   Purchase_IsWeekend          3660 non-null   int64  
 8   Category_Books              3660 non-null   int64  
 9   Category_Clothing           3660 non-null   int64  
 10  Category_Electronics        3660 non-null   int64  
 11  Category_Home & Kitchen     3660 non-null   int64  
 12  Category_Sports             3660 non-null   int64  
 13  Category_Toys               3660 

In [ ]:
# Lưu dữ liệu sạch xuống thư mục processed
output_path = os.path.join(project_root, 'data', 'processed', 'ecommerce_features.csv')
df_final.to_csv(output_path, index=False)
print(f"Dữ liệu đã được lưu thành công tại: {output_path}")

Dữ liệu đã được lưu thành công tại: e:\PRE2\Predicting_Product_Sales\data\processed\ecommerce_features.csv
